# User-anchored local ARG traceback

This notebook loads an inferred and dated `.trees` file plus phased VCF observations, creates and saves a synthetic/full ARG with explicit routing nodes, reloads it into a `FastARGTrace`, and resolves a user-supplied genomic interval and event time. It initializes VCF partial likelihoods on the existing `ARGState`, samples a prior-driven stepwise local ARG, computes its whole-chromosome JC69 likelihood and posterior reward, splices it into the fixed chromosome, removes temporary source routing nodes, saves a clean refined `.trees` file, and independently rescores the export. The principal returned variables are `variant_data`, `local_environment`, `local_initial_state`, `local_sample_batch`, `local_proposal`, `local_splice_result`, and `refined_tree_sequence`.

## 1. Imports and workspace setup

In [1]:
from pathlib import Path
import sys

import numpy as np
import tskit


def find_workspace_root(start=None):
    start = Path.cwd() if start is None else Path(start).expanduser().resolve()
    fallback = Path("/Users/pratik/Documents/work/aim3/simpliied")
    for candidate in (start, *start.parents, fallback):
        if (candidate / "arg/new_rl/trace.py").is_file():
            return candidate
    raise RuntimeError("Could not locate the workspace root")


WORKSPACE_ROOT = find_workspace_root()
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

from arg.new_rl import (
    LocalRefinementRequest,
    LocalSamplingConfig,
    PreparedLocalRefinement,
    SimpleARGEnvironment,
    build_fast_trace_from_full_arg,
    build_synthetic_full_arg,
    compute_tree_sequence_vcf_log_likelihood,
    export_refined_tree_sequence,
    initialize_local_arg_state,
    sample_local_trajectories,
    splice_local_proposal,
    trace_local_dependencies,
)
from arg.new_rl.trace import EVENT_KIND_COALESCENCE
from arg.utils import load_vcf_variants

WORKSPACE_ROOT

PosixPath('/Users/pratik/Documents/work/aim3/simpliied')

## 2. Choose the input and saved synthetic ARG paths

Edit `INPUT_TREES_PATH` and `INPUT_VCF_PATH` for a different inferred ARG and its phased observations. The source file is never overwritten.

In [2]:
INPUT_TREES_PATH = (
    WORKSPACE_ROOT / "arg/validation/output/tsinfer/l1mb_dated.trees"
)
INPUT_VCF_PATH = (
    WORKSPACE_ROOT / "arg/validation/vcf/sim_l1mb_0.vcf"
)
SYNTHETIC_TREES_PATH = INPUT_TREES_PATH.with_name(
    f"{INPUT_TREES_PATH.stem}_local_refinement_synthetic_full_arg.trees"
)
REFINED_TREES_PATH = INPUT_TREES_PATH.with_name(
    f"{INPUT_TREES_PATH.stem}_local_refined.trees"
)
OVERWRITE_REFINED_OUTPUT = True

if not INPUT_TREES_PATH.is_file():
    raise FileNotFoundError(INPUT_TREES_PATH)
if not INPUT_VCF_PATH.is_file():
    raise FileNotFoundError(INPUT_VCF_PATH)
if SYNTHETIC_TREES_PATH.resolve() == INPUT_TREES_PATH.resolve():
    raise ValueError("The synthetic output path must differ from the input path")
if REFINED_TREES_PATH.resolve() in {
    INPUT_TREES_PATH.resolve(),
    SYNTHETIC_TREES_PATH.resolve(),
}:
    raise ValueError("The refined output path must be a separate file")

{
    "input": str(INPUT_TREES_PATH),
    "vcf": str(INPUT_VCF_PATH),
    "synthetic_output": str(SYNTHETIC_TREES_PATH),
    "refined_output": str(REFINED_TREES_PATH),
}

{'input': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/output/tsinfer/l1mb_dated.trees',
 'vcf': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/vcf/sim_l1mb_0.vcf',
 'synthetic_output': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/output/tsinfer/l1mb_dated_local_refinement_synthetic_full_arg.trees',
 'refined_output': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/output/tsinfer/l1mb_dated_local_refined.trees'}

## 3. Load, add synthetic nodes, and save the derived `.trees` file

In [3]:
input_tree_sequence = tskit.load(str(INPUT_TREES_PATH))
synthetic_conversion = build_synthetic_full_arg(
    input_tree_sequence,
    split_rule="balanced",
    ensure_unique_event_times=True,
)
synthetic_tree_sequence = synthetic_conversion.tree_sequence
synthetic_tree_sequence.dump(str(SYNTHETIC_TREES_PATH))

conversion_summary = {
    "input_nodes": input_tree_sequence.num_nodes,
    "input_edges": input_tree_sequence.num_edges,
    "synthetic_nodes": synthetic_tree_sequence.num_nodes,
    "synthetic_edges": synthetic_tree_sequence.num_edges,
    "inserted_recombination_nodes": synthetic_conversion.metadata[
        "synthetic_recombination_node_count"
    ],
    "saved_path": str(SYNTHETIC_TREES_PATH),
}
conversion_summary

{'input_nodes': 483,
 'input_edges': 1703,
 'synthetic_nodes': 3135,
 'synthetic_edges': 7964,
 'inserted_recombination_nodes': 2652,
 'saved_path': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/output/tsinfer/l1mb_dated_local_refinement_synthetic_full_arg.trees'}

## 4. Reload the synthetic ARG and build `FastARGTrace`

The displayed `valid_event_time_range` is the inclusive range allowed by the user-input cell below.

In [4]:
reloaded_synthetic_tree_sequence = tskit.load(str(SYNTHETIC_TREES_PATH))
arg_trace = build_fast_trace_from_full_arg(
    reloaded_synthetic_tree_sequence,
    require_unique_event_times=True,
    allow_no_recombination=True,
)

coalescence_event_times = arg_trace.event_time[
    arg_trace.event_kind == EVENT_KIND_COALESCENCE
]
if coalescence_event_times.size == 0:
    raise ValueError("The trace contains no coalescence events")

highest_coalescent_event_time = float(coalescence_event_times.max())
valid_event_time_range = (0.0, highest_coalescent_event_time)

{
    "sequence_length": arg_trace.sequence_length,
    "trace_events": arg_trace.event_count,
    "recombination_events": arg_trace.recombination_event_count,
    "coalescence_events": arg_trace.coalescence_event_count,
    "valid_event_time_range": valid_event_time_range,
}

{'sequence_length': 1000000.0,
 'trace_events': 1801,
 'recombination_events': 1326,
 'coalescence_events': 475,
 'valid_event_time_range': (0.0, 74872.68528957557)}

## 5. User input: local genomic range and event time

Edit these values, then run this cell and the remaining cells. `USER_LOCAL_RANGE` is half-open `[left, right)`. `USER_EVENT_TIME` may be any numeric time from zero through `highest_coalescent_event_time`; it does not need to equal an existing event exactly. Population size and recombination parameters define the CWR prior, while `MUTATION_RATE` defines JC69 branch lengths. These values are not inferred from the `.trees` file. The VCF-aware workflow constructs an exact compact local grid and does not allocate one likelihood row per base.

In [5]:
highest_coalescent_event_time

74872.68528957557

In [6]:
USER_LOCAL_RANGE = (14000.0, 190000.0)
USER_EVENT_TIME = min(25_000.0, highest_coalescent_event_time)
POPULATION_SIZE = 10_000.0
MUTATION_RATE = 2e-8
RECOMBINATION_RATE = 2e-8
RHO = None  # Set a float to override 4 * Ne * r * sequence_length.
REWARD_C = 30_000.0

local_left, local_right = map(float, USER_LOCAL_RANGE)
event_time = float(USER_EVENT_TIME)

if not 0.0 <= local_left < local_right <= arg_trace.sequence_length:
    raise ValueError(
        "USER_LOCAL_RANGE must satisfy "
        f"0 <= left < right <= {arg_trace.sequence_length}"
    )
if not 0.0 <= event_time <= highest_coalescent_event_time:
    raise ValueError(
        "USER_EVENT_TIME must be within the inclusive range "
        f"[0, {highest_coalescent_event_time}]"
    )

{
    "local_range": (local_left, local_right),
    "event_time": event_time,
    "population_size": POPULATION_SIZE,
    "mutation_rate": MUTATION_RATE,
    "recombination_rate": RECOMBINATION_RATE,
    "rho_override": RHO,
    "reward_C": REWARD_C,
}

{'local_range': (14000.0, 190000.0),
 'event_time': 25000.0,
 'population_size': 10000.0,
 'mutation_rate': 2e-08,
 'recombination_rate': 2e-08,
 'rho_override': None,
 'reward_C': 30000.0}

## 6. Trace to the requested cut and build the local context

A numeric time resolves to the state immediately before the first trace event whose time is greater than or equal to the requested value.

In [7]:
local_refinement_request = LocalRefinementRequest(
    genomic_range=(local_left, local_right),
    cut_time=event_time,
)

# Main returned object for downstream local reconstruction code.
local_refinement_context = trace_local_dependencies(
    arg_trace,
    local_refinement_request,
)
prepared_local_refinement = PreparedLocalRefinement(
    source_tree_sequence=input_tree_sequence,
    synthetic_conversion=synthetic_conversion,
    trace=arg_trace,
    context=local_refinement_context,
)

# Full FastARGState at the same resolved time boundary.
traceback_state = arg_trace.initial_state().advance_to(
    local_refinement_context.resolved_cut.cut_step
)

# Convenient aliases for notebook exploration.
local_active_lineages = local_refinement_context.cut_active_lineages
local_promoted_dependencies = (
    local_refinement_context.promoted_dependency_lineages
)
local_fixed_boundary = local_refinement_context.boundary_attachments

{
    "returned_object_variable": "local_refinement_context",
    "trace_state_variable": "traceback_state",
    "status": local_refinement_context.status,
    "resolved_cut_step": local_refinement_context.resolved_cut.cut_step,
    "active_lineages_at_cut": len(local_active_lineages),
}

{'returned_object_variable': 'local_refinement_context',
 'trace_state_variable': 'traceback_state',
 'status': 'valid',
 'resolved_cut_step': 1628,
 'active_lineages_at_cut': 54}

## 7. Inspect the returned object

In [8]:
result_summary = {
    "status": local_refinement_context.status,
    "requested_range": local_refinement_context.request.genomic_range,
    "requested_time": local_refinement_context.request.cut_time,
    "resolved_cut_step": local_refinement_context.resolved_cut.cut_step,
    "state_current_time": traceback_state.current_time,
    "next_event_time": local_refinement_context.resolved_cut.next_event_time,
    "cut_active_lineage_ids": tuple(
        lineage.node_id for lineage in local_active_lineages
    ),
    "promoted_dependency_ids": tuple(
        lineage.node_id for lineage in local_promoted_dependencies
    ),
    "selected_event_indices": (
        local_refinement_context.selected_event_indices
    ),
    "complexity": dict(local_refinement_context.complexity),
    "diagnostics": local_refinement_context.rejection_diagnostics,
}

result_summary

{'status': 'valid',
 'requested_range': (14000.0, 190000.0),
 'requested_time': 25000.0,
 'resolved_cut_step': 1628,
 'state_current_time': 24819.75424761958,
 'next_event_time': 25029.56260142321,
 'cut_active_lineage_ids': (243,
  247,
  311,
  312,
  313,
  325,
  326,
  364,
  371,
  376,
  385,
  496,
  500,
  891,
  1241,
  1454,
  2221,
  2222,
  2225,
  2229,
  2236,
  2239,
  2240,
  2251,
  2257,
  2266,
  2272,
  2602,
  2608,
  2615,
  2616,
  2620,
  2624,
  2625,
  2626,
  2629,
  2637,
  2638,
  2697,
  2831,
  2833,
  2835,
  2837,
  2838,
  2858,
  2861,
  2989,
  2991,
  2992,
  2995,
  2996,
  2997,
  3023,
  3024),
 'promoted_dependency_ids': (310,
  365,
  366,
  378,
  381,
  382,
  383,
  384,
  386,
  387,
  388,
  389,
  390,
  391,
  392,
  393,
  394,
  2603,
  2604,
  3079,
  3080,
  3081,
  3082,
  3083,
  3084,
  3111,
  3113,
  3125,
  3127,
  3128,
  3129),
 'selected_event_indices': (1642,
  1643,
  1644,
  1645,
  1649,
  1663,
  1666,
  1668,
  1669,


In [9]:
active_lineage_rows = [
    {
        "node_id": lineage.node_id,
        "role": lineage.role,
        "mutable_segments": lineage.mutable_segments,
        "fixed_segments": lineage.fixed_segments,
        "first_active_step": lineage.first_active_step,
        "first_active_time": lineage.first_active_time,
    }
    for lineage in local_refinement_context.active_lineages
]

active_lineage_rows

[{'node_id': 243,
  'role': 'mutable_target',
  'mutable_segments': ((187128.0, 190000.0),),
  'fixed_segments': ((190000.0, 190275.0),),
  'first_active_step': 1602,
  'first_active_time': 22623.97639003238},
 {'node_id': 247,
  'role': 'mutable_target',
  'mutable_segments': ((142366.0, 153347.0),),
  'fixed_segments': (),
  'first_active_step': 1524,
  'first_active_time': 18086.84126678574},
 {'node_id': 311,
  'role': 'mutable_target',
  'mutable_segments': ((106717.0, 110487.0),),
  'fixed_segments': (),
  'first_active_step': 1276,
  'first_active_time': 10079.939171716485},
 {'node_id': 312,
  'role': 'mutable_target',
  'mutable_segments': ((126101.0, 127385.0),),
  'fixed_segments': (),
  'first_active_step': 1295,
  'first_active_time': 10303.3395747657},
 {'node_id': 313,
  'role': 'mutable_target',
  'mutable_segments': ((75844.0, 83367.0),),
  'fixed_segments': (),
  'first_active_step': 1412,
  'first_active_time': 13322.09994190211},
 {'node_id': 325,
  'role': 'mutable

In [10]:
# Initialize VCF partials on the shared ARGState and sample one complete prior history.
variant_data = load_vcf_variants(INPUT_VCF_PATH)
local_environment = SimpleARGEnvironment(
    variant_data=variant_data,
    population_size=POPULATION_SIZE,
    mutation_rate=MUTATION_RATE,
    recombination_rate=RECOMBINATION_RATE,
    rho=RHO,
    reward_C=REWARD_C,
)
local_initial_state = initialize_local_arg_state(
    prepared_local_refinement,
    local_environment,
)
local_lineage_likelihood_summary = [
    {
        "node_id": lineage.node_id,
        "material_blocks": lineage.material_segments.segments,
        "vcf_variant_indices": lineage.variant_indices,
        "partial_shape": tuple(lineage.partials.shape),
        "partial_preview": lineage.partials[:3].detach().cpu().tolist(),
    }
    for lineage in local_initial_state.active_lineages
]
local_sample_batch = sample_local_trajectories(
    prepared_local_refinement,
    local_environment,
    LocalSamplingConfig(
        sample_count=1,
        seed=1,
    ),
    initial_state=local_initial_state,
)
if not local_sample_batch.proposals:
    raise RuntimeError(local_sample_batch.diagnostics)

local_proposal = local_sample_batch.proposals[0]
{
    "proposal_events": len(local_proposal.events),
    "proposal_nodes": len(local_proposal.nodes),
    "proposal_edges": len(local_proposal.edges),
    "prior_log_probability": local_proposal.prior_log_probability,
    "whole_chromosome_vcf_log_likelihood": local_proposal.log_likelihood,
    "terminal_log_reward": local_proposal.log_reward,
    "target_vcf_row_count": len(local_initial_state.target_variant_indices),
    "outside_log_likelihood": local_initial_state.outside_log_likelihood,
    "initial_lineage_likelihood_preview": local_lineage_likelihood_summary[:5],
    "root_region_count": len(local_proposal.root_intervals),
    "root_intervals_preview": local_proposal.root_intervals[:10],
    "topology_digest": local_proposal.topology_digest,
}

{'proposal_events': 295,
 'proposal_nodes': 515,
 'proposal_edges': 621,
 'prior_log_probability': -2384.5898060613736,
 'whole_chromosome_vcf_log_likelihood': -22458.09835283432,
 'terminal_log_reward': 5157.311841104306,
 'target_vcf_row_count': 292,
 'outside_log_likelihood': -19374.186020424742,
 'initial_lineage_likelihood_preview': [{'node_id': 0,
   'material_blocks': ((40, 58), (64, 67)),
   'vcf_variant_indices': (57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    78,
    79),
   'partial_shape': (18, 4),
   'partial_preview': [[0.0, 0.0, 0.0, 1.0],
    [1.0, 0.0, 0.0, 0.0],
    [1.0, 0.0, 0.0, 0.0]]},
  {'node_id': 3,
   'material_blocks': ((259, 267),),
   'vcf_variant_indices': (244, 245, 246, 247, 248, 249),
   'partial_shape': (6, 4),
   'partial_preview': [[1.0, 0.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 0.0],
    [0.0, 0.0, 1.0, 0.0]]},
  {'node_id': 5,
   'material_blocks': ((58, 64),),
   'vcf_varia

In [11]:
# Splice only authorized local intervals, remove source synthetic nodes, validate,
# and save a clean chromosome-wide tree sequence at a separate output path.
local_splice_result = splice_local_proposal(
    prepared_local_refinement,
    local_proposal,
)
if not local_splice_result.validation.is_valid:
    raise RuntimeError(local_splice_result.validation.errors)

refined_tree_sequence = local_splice_result.refined_tree_sequence
saved_refined_path = export_refined_tree_sequence(
    local_splice_result,
    REFINED_TREES_PATH,
    overwrite=OVERWRITE_REFINED_OUTPUT,
)
reloaded_refined_tree_sequence = tskit.load(str(saved_refined_path))
independent_export_log_likelihood = compute_tree_sequence_vcf_log_likelihood(
    reloaded_refined_tree_sequence,
    variant_data,
    mutation_rate=MUTATION_RATE,
)
likelihood_parity_error = (
    independent_export_log_likelihood - local_proposal.log_likelihood
)
if not np.isclose(likelihood_parity_error, 0.0, atol=1e-7, rtol=0.0):
    raise RuntimeError(
        "Incremental and independently rescored likelihoods differ: "
        f"{likelihood_parity_error}"
    )

{
    "saved_refined_path": str(saved_refined_path),
    "validation": local_splice_result.validation,
    "removed_source_synthetic_nodes": len(
        local_splice_result.removed_source_synthetic_node_ids
    ),
    "retained_local_nodes": len(local_splice_result.local_node_id_map),
    "refined_nodes": refined_tree_sequence.num_nodes,
    "refined_edges": refined_tree_sequence.num_edges,
    "proposal_log_likelihood": local_proposal.log_likelihood,
    "independent_export_log_likelihood": independent_export_log_likelihood,
    "likelihood_parity_error": likelihood_parity_error,
}

{'saved_refined_path': '/Users/pratik/Documents/work/aim3/simpliied/arg/validation/output/tsinfer/l1mb_dated_local_refined.trees',
 'validation': LocalValidationReport(is_valid=True, errors=(), warnings=(), counts={'source_node_count': 483, 'refined_node_count': 998, 'refined_edge_count': 2268, 'removed_source_synthetic_node_count': 2652, 'retained_local_node_count': 515, 'sampled_event_count': 295, 'local_root_region_count': 176, 'dangling_target_node_count': 0, 'dangling_target_tree_count': 0, 'target_genotypes_preserved': True, 'collapsed_local_recombination_parity': True, 'exterior_unchanged': True, 'likelihood_parity': True, 'independent_vcf_log_likelihood': -22458.098352834317}),
 'removed_source_synthetic_nodes': 2652,
 'retained_local_nodes': 515,
 'refined_nodes': 998,
 'refined_edges': 2268,
 'proposal_log_likelihood': -22458.09835283432,
 'independent_export_log_likelihood': -22458.098352834317,
 'likelihood_parity_error': 3.637978807091713e-12}